In [1]:
# %%
# ═══════════════════════════════════════════════════════════
#  SmolVLM2-2.2B  ×  Kvasir-VQA  —  Fine-Tuned Model Testing
# ═══════════════════════════════════════════════════════════

import os


# ── SET ALL PATHS FIRST ───────────────────────────────────
cache_path = "/mnt/d/huggingface_cache"
datasets_path = os.path.join(cache_path, "datasets")
tmp_path      = "/mnt/d/tmp"

os.environ["HF_HOME"]            = cache_path
os.environ["HF_DATASETS_CACHE"]  = datasets_path
os.environ["TRANSFORMERS_CACHE"] = os.path.join(cache_path, "transformers")
os.environ["HF_HUB_CACHE"]       = os.path.join(cache_path, "hub")
os.environ["TMPDIR"]             = tmp_path
os.environ["TEMP"]               = tmp_path
os.environ["TMP"]                = tmp_path   

import glob
import torch
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel
import numpy as np
from sklearn.metrics import accuracy_score
from rouge_score import rouge_scorer
import json
from tqdm import tqdm
from collections import defaultdict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /home/akanksha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
# ── CONFIG ────────────────────────────────────────────────
MODEL_ID   = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
FINETUNED_DIR   = "/mnt/d/workspace/Lama_EndoscopyQA/smolvlm2-kvasir-finetuned"   # your saved adapter path
checkpoint_dirs =  glob.glob(os.path.join(FINETUNED_DIR, "checkpoint-*"))
CHECKPOINT_DIR = checkpoint_dirs[0]
MAX_NEW_TOKENS  = 128       # answer is usually short for VQA
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# Verify contents
print(f"Files in checkpoint:{CHECKPOINT_DIR}")
for f in os.listdir(CHECKPOINT_DIR):
    print(f"  {f}")

Files in checkpoint:/mnt/d/workspace/Lama_EndoscopyQA/smolvlm2-kvasir-finetuned/checkpoint-900
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  optimizer.pt
  README.md
  rng_state.pth
  scheduler.pt
  tokenizer.json
  tokenizer_config.json
  trainer_state.json
  training_args.bin


In [7]:
from peft import PeftModel

# Same base model + processor
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # same bnb_config as training
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Load your LoRA adapter on top
FINETUNED_DIR = "/mnt/d/workspace/Lama_EndoscopyQA/smolvlm2-kvasir-finetuned"
checkpoint_dir = os.path.join(
    FINETUNED_DIR,
    next(d for d in os.listdir(FINETUNED_DIR) if d.startswith("checkpoint-"))
)
print(checkpoint_dir)
model = PeftModel.from_pretrained(model, checkpoint_dir)
model.eval()

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

/mnt/d/workspace/Lama_EndoscopyQA/smolvlm2-kvasir-finetuned/checkpoint-900


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): SmolVLMForConditionalGeneration(
      (model): SmolVLMModel(
        (vision_model): SmolVLMVisionTransformer(
          (embeddings): SmolVLMVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(729, 1152)
          )
          (encoder): SmolVLMEncoder(
            (layers): ModuleList(
              (0-26): 27 x SmolVLMEncoderLayer(
                (self_attn): SmolVLMVisionAttention(
                  (k_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=1152, out_features=1152, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1152, out_features=32, bias=False)
                    )
               

In [ ]:
# # ── 1. LOAD PROCESSOR ─────────────────────────────────────

# processor = AutoProcessor.from_pretrained(FINETUNED_DIR)
# print("✅ Processor loaded!")

# # ── 2. LOAD BASE MODEL IN 4-BIT ───────────────────────────
# print("🧠 Loading base model in 4-bit...")
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# base_model = AutoModelForImageTextToText.from_pretrained(
#     BASE_MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )

# # ── 3. LOAD FINE-TUNED LORA ADAPTER ───────────────────────
# print(f"🔧 Loading LoRA adapter from {FINETUNED_DIR}...")
# model = PeftModel.from_pretrained(base_model, FINETUNED_DIR)
# model.eval()
# print("✅ Fine-tuned model ready!")

✅ Processor loaded!
🧠 Loading base model in 4-bit...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

🔧 Loading LoRA adapter from /mnt/d/workspace/Lama_EndoscopyQA/smolvlm2-kvasir-finetuned...
✅ Fine-tuned model ready!


In [8]:

SYSTEM_INSTRUCTION = (
 "You are a medical AI assistant specialized in gastrointestinal endoscopy. "
    "Analyze the provided endoscopic image carefully and answer the clinical question. "
    "Answer in keywords or short phrases only. "
    "Questions based on: anatomical landmarks, pathological findings, instrument presence, image quality"
    "Examples for answers: '0', 'colonoscopy', 'none', 'yes', 'center'."
)
# ── 4. INFERENCE HELPER ───────────────────────────────────
def predict(image: Image.Image, question: str) -> str:
    """Run a single VQA inference and return the model's answer."""
    messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": question},   # plain question only
        ],
    },
]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=prompt,
        images=[[image.convert("RGB")]],
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,       # greedy — deterministic for evaluation
        )

    # Decode only the newly generated tokens (strip the prompt)
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()

In [9]:
# ── 5. LOAD TEST DATA ─────────────────────────────────────
print("\n📂 Loading Kvasir-VQA test samples...")
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]
 
# Use the SAME seed=42 shuffle as training so index ranges are consistent:
#   [0 : TRAIN_SAMPLES]                          → train
#   [TRAIN_SAMPLES : TRAIN_SAMPLES+EVAL_SAMPLES] → eval
#   [TRAIN_SAMPLES+EVAL_SAMPLES : ...]            → test  ← we use this
TRAIN_SAMPLES = 5000
EVAL_SAMPLES  = 100
TEST_SAMPLES  = 5000
 
ds = ds.shuffle(seed=42)   # identical seed to training script
test_start = TRAIN_SAMPLES + EVAL_SAMPLES          # 550
test_end   = test_start + TEST_SAMPLES             # 650
test_ds    = ds.select(range(test_start, test_end))
 
print(f"✅ Testing on {len(test_ds)} samples (indices {test_start}–{test_end - 1})")
 


📂 Loading Kvasir-VQA test samples...


Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Testing on 5000 samples (indices 5100–10099)


In [10]:
# ── Preview dataset structure ─────────────────────────────
print("── Dataset columns:", test_ds.column_names)
# print("── Sample categories:", dataset.unique("source"))
# print("── Sample questions:",  dataset.unique("question")[:5], "\n")

── Dataset columns: ['image', 'source', 'question', 'answer', 'img_id']


In [11]:
# Map semantically equivalent answers to a canonical form
ANSWER_SYNONYMS = {
    "none":     ["none", "not present", "absent", "nothing",  "zero", "there are none", "no instruments"],
    "yes":      ["yes", "correct", "true", "present", "there is", "i can see"],
    "no":       [ "false", "not", "none", "absent", "cannot see"],
    "normal":   ["normal", "no abnormality", "no findings", "unremarkable"],
    "colonoscopy": ["colonoscopy", "colon", "colonoscopic"],
}

import re

def normalize_answer(text: str) -> str:
    text = text.lower().strip()
    for canonical, synonyms in ANSWER_SYNONYMS.items():
        for syn in synonyms:
            # \b ensures we match whole words only, not substrings
            if re.search(rf"\b{re.escape(syn)}\b", text):
                return canonical
    return text

# ── 6. BATCH EVALUATION ───────────────────────────────────
print("\n🔍 Running inference...")

predictions = []
ground_truths = []
results = []

for sample in tqdm(test_ds, desc="Evaluating"):
    image    = sample["image"]
    question = sample["question"]
    gt_answer = str(sample["answer"]).strip().lower()
    source    = sample["source"]
    img_id    = sample["img_id"]

    pred_answer = predict(image, question).lower()

    gt_norm   = normalize_answer(gt_answer)
    pred_norm = normalize_answer(pred_answer)
    correct   = (gt_norm == pred_norm) or (gt_norm in pred_answer.lower())

    predictions.append(pred_answer)
    ground_truths.append(gt_answer)
    results.append({
        "img_id":    img_id,
        "source":    source,
        "question":  question,
        "gt_answer": gt_answer,
        "gt_norm":   gt_norm,
        "predicted": pred_answer,
        "pred_norm": pred_norm,
        "correct":   correct,
    })



🔍 Running inference...


Evaluating: 100%|██████████| 5000/5000 [55:15<00:00,  1.51it/s]  


In [12]:

# ── ROUGE 1 and 2 ──
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for gt, pred in zip(ground_truths, predictions):
    scores = scorer.score(gt, pred)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

avg_rouge1 = np.mean(rouge1_scores)
avg_rouge2 = np.mean(rouge2_scores)
avg_rougeL = np.mean(rougeL_scores)

# ── BLEU ──
smoothie = SmoothingFunction().method1
bleu_scores = [
    sentence_bleu(
        [gt.split()],          # reference (list of list of words)
        pred.split(),          # hypothesis (list of words)
        smoothing_function=smoothie
    )
    for gt, pred in zip(ground_truths, predictions)
]
avg_bleu = np.mean(bleu_scores)

# ── METEOR ──
meteor_scores = [
    meteor_score([gt.split()], pred.split())
    for gt, pred in zip(ground_truths, predictions)
]
avg_meteor = np.mean(meteor_scores)
# Fixed (proper EM with normalization)
strict_matches = [
    normalize_answer(r["gt_answer"]) == normalize_answer(r["predicted"]) 
    for r in results
]
strict_accuracy = np.mean(strict_matches)

# ── PRINT ALL ──
print(f"  ROUGE-1   : {np.mean(rouge1_scores):.4f}")
print(f"  ROUGE-2   : {np.mean(rouge2_scores):.4f}")
print(f"  ROUGE-L   : {np.mean(rougeL_scores):.4f}")
print(f"  BLEU      : {np.mean(bleu_scores):.4f}")
print(f"  METEOR    : {np.mean(meteor_scores):.4f}")
print(f"  Strict Exact Match   : {strict_accuracy:.2%}  ({sum(strict_matches)}/{len(strict_matches)})")


  ROUGE-1   : 0.8742
  ROUGE-2   : 0.1792
  ROUGE-L   : 0.8717
  BLEU      : 0.2022
  METEOR    : 0.4993
  Strict Exact Match   : 82.16%  (4108/5000)


In [14]:
# # Normalised accuracy (primary metric)
# norm_correct    = [r["correct"] for r in results]
# norm_accuracy   = np.mean(norm_correct)
 
# # Strict exact match on raw strings (secondary)
# strict_matches  = [r["gt_answer"] == r["predicted"] for r in results]
# strict_accuracy = np.mean(strict_matches)
 
# # ROUGE-L on raw strings (partial-credit)
# scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
# rouge_scores = [
#     scorer.score(gt, pred)["rougeL"].fmeasure
#     for gt, pred in zip(ground_truths, predictions)
# ]
# avg_rouge_l = np.mean(rouge_scores)
 
# Per-source breakdown
source_stats = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results:
    source_stats[r["source"]]["total"]   += 1
    source_stats[r["source"]]["correct"] += int(r["correct"])
 
# ── 8. PRINT RESULTS ──────────────────────────────────────
print("\n" + "=" * 55)
print("  EVALUATION RESULTS")
print("=" * 55)
#print(f"  Normalised Accuracy  : {norm_accuracy:.2%}  ({sum(norm_correct)}/{len(norm_correct)})")
print(f"  Strict Exact Match   : {strict_accuracy:.2%}  ({sum(strict_matches)}/{len(strict_matches)})")
print("-" * 55)
print(f"  ROUGE-1              : {avg_rouge1:.4f}")
print(f"  ROUGE-2              : {avg_rouge2:.4f}")
print(f"  ROUGE-L              : {avg_rougeL:.4f}")
print(f"  BLEU                 : {avg_bleu:.4f}")
print(f"  METEOR               : {avg_meteor:.4f}")
print("-" * 55)
print("  Per-Source Breakdown:")
for src, stat in sorted(source_stats.items()):
    acc = stat["correct"] / stat["total"]
    print(f"    {src:<30} {acc:.2%}  ({stat['correct']}/{stat['total']})")
print("=" * 55)
 
 
# ── 9. QUALITATIVE SAMPLES ────────────────────────────────
print("\n🔎 Sample Predictions (first 10):\n")
header = f"{ 'Question':<40} {'GT':>12} {'GT-norm':>10} {'Pred':>12} {'Pred-norm':>10} {'OK'}"
print(header)
print("-" * len(header))
for r in results[:10]:
    icon = "✅" if r["correct"] else "❌"
    print(
        #f"{str(r['img_id']):<12} "
        f"{r['question'][:39]:<40} "
        f"{r['gt_answer'][:11]:>12} "
        f"{r['gt_norm'][:9]:>10} "
        f"{r['predicted'][:11]:>12} "
        f"{r['pred_norm'][:9]:>10} "
        f"{icon}"
    )
 
 #
# ── 10. SAVE RESULTS ──────────────────────────────────────
output_path = os.path.join(FINETUNED_DIR, "eval_results.json")
summary = {
    "total_samples":       len(results),
    #"normalised_accuracy": round(float(norm_accuracy), 4),
    "strict_exact_match":  round(float(strict_accuracy), 4),
    "rouge1":              round(float(avg_rouge1), 4),
    "rouge2":              round(float(avg_rouge2), 4),
    "rougeL":              round(float(avg_rougeL), 4),
    "bleu":                round(float(avg_bleu), 4),
    "meteor":              round(float(avg_meteor), 4),
    "per_source": {
        src: {"accuracy": round(s["correct"] / s["total"], 4), **s}
        for src, s in source_stats.items()
    },
    "per_sample_results":  results,
}
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n💾 Full results saved to: {output_path}")
 
 



  EVALUATION RESULTS
  Strict Exact Match   : 82.16%  (4108/5000)
-------------------------------------------------------
  ROUGE-1              : 0.8742
  ROUGE-2              : 0.1792
  ROUGE-L              : 0.8717
  BLEU                 : 0.2022
  METEOR               : 0.4993
-------------------------------------------------------
  Per-Source Breakdown:
    Esophagitis                    90.72%  (1281/1412)
    Instrument                     71.18%  (541/760)
    Normal                         100.00%  (218/218)
    Polyps                         72.68%  (891/1226)
    Ulcerative Colitis             91.76%  (1270/1384)

🔎 Sample Predictions (first 10):

Question                                           GT    GT-norm         Pred  Pred-norm OK
-------------------------------------------------------------------------------------------
How many instrumnets are in the image?              0          0            0          0 ✅
Does this image contain any finding?               no   

In [24]:
# Check distribution in your training slice
from collections import Counter
ds_shuffled = ds.shuffle(seed=42)
train_slice = ds_shuffled.select(range(500))
print(Counter(train_slice["answer"]))  # are answers balanced?

Counter({'none': 100, 'no': 75, 'yes': 52, '0': 45, '1': 34, 'not relevant': 19, 'colonoscopy': 19, 'oesophagitis': 13, 'polyp': 12, 'gastroscopy': 11, 'pink; red; white': 10, 'z-line': 9, 'center; center-left; center-right; lower-center; lower-left; lower-right; upper-center; upper-left; upper-right': 8, 'tube': 8, '2': 7, 'ulcerative colitis': 5, 'paris iia': 5, 'paris is': 5, 'paris ip': 4, 'pink': 3, '5-10mm': 3, '>20mm': 3, 'flesh; red': 2, 'center; upper-center': 2, 'center': 2, 'pink; red': 2, 'pink; white': 2, '11-20mm': 2, '<5mm': 2, 'center; center-left; lower-center; lower-left; upper-center; upper-left': 2, 'center-left; center-right; upper-center; upper-left': 1, 'polyp snare; tube': 1, 'center; center-left; upper-left': 1, 'upper-right': 1, 'center; center-left; center-right; lower-center; lower-right; upper-center; upper-left; upper-right': 1, 'center; upper-center; upper-left; upper-right': 1, 'orange; brown': 1, 'center; center-left; center-right; lower-center; lower-l